In [2]:
import pandas as pd

df = pd.read_csv("data/flights_sample.csv")

df.head()

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,4,7,2,EV,4900,N759EV,FWA,DTW,1340,...,1423.0,-13.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,2015,1,24,6,AS,611,N413AS,LAS,SEA,1910,...,2133.0,-12.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,2015,7,8,3,WN,1483,N463WN,OAK,SEA,630,...,812.0,-8.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,2015,5,26,2,WN,193,N7745A,STL,DAL,810,...,1222.0,152.0,0,0,NaN,0.0,0.0,0.0,152.0,0.0
4,2015,7,6,1,UA,253,N213UA,IAH,HNL,1000,...,1316.0,-2.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
import pandas as pd

df = pd.read_csv("data/flights_sample.csv")

df.shape

(581908, 31)

In [4]:
delay_columns = [
    "DEPARTURE_DELAY",
    "ARRIVAL_DELAY",
    "AIR_SYSTEM_DELAY",
    "SECURITY_DELAY",
    "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "WEATHER_DELAY"
]

df[delay_columns + ["CANCELLED", "CANCELLATION_REASON"]].isnull().sum()

DEPARTURE_DELAY          8640
ARRIVAL_DELAY           10553
AIR_SYSTEM_DELAY       475550
SECURITY_DELAY         475550
AIRLINE_DELAY          475550
LATE_AIRCRAFT_DELAY    475550
WEATHER_DELAY          475550
CANCELLED                   0
CANCELLATION_REASON    572856
dtype: int64

In [5]:
df["CANCELLED"].value_counts()

CANCELLED
0    572856
1      9052
Name: count, dtype: int64

In [6]:
cancelled_df = df[df["CANCELLED"] == 1]
active_df = df[df["CANCELLED"] == 0]

print("Cancelled flights:", cancelled_df.shape)
print("Active flights:", active_df.shape)

Cancelled flights: (9052, 31)
Active flights: (572856, 31)


In [7]:
active_df[delay_columns] = active_df[delay_columns].fillna(0)

active_df[delay_columns].isnull().sum()

DEPARTURE_DELAY        0
ARRIVAL_DELAY          0
AIR_SYSTEM_DELAY       0
SECURITY_DELAY         0
AIRLINE_DELAY          0
LATE_AIRCRAFT_DELAY    0
WEATHER_DELAY          0
dtype: int64

In [8]:
df["CANCELLATION_REASON"] = df["CANCELLATION_REASON"].fillna("Not Cancelled")

df["CANCELLATION_REASON"].value_counts()

CANCELLATION_REASON
Not Cancelled    572856
B                  4890
A                  2543
C                  1616
D                     3
Name: count, dtype: int64

In [9]:
cleaned_df = pd.concat([active_df, cancelled_df])

cleaned_df.shape

(581908, 31)

In [10]:
cleaned_df[delay_columns].isnull().sum()

DEPARTURE_DELAY        8640
ARRIVAL_DELAY          9052
AIR_SYSTEM_DELAY       9052
SECURITY_DELAY         9052
AIRLINE_DELAY          9052
LATE_AIRCRAFT_DELAY    9052
WEATHER_DELAY          9052
dtype: int64

In [11]:
cleaned_df["MONTH_NAME"] = pd.to_datetime(
    cleaned_df["MONTH"], format="%m"
).dt.month_name()

cleaned_df[["MONTH", "MONTH_NAME"]].head()

,MONTH,MONTH_NAME
0,4,April
1,1,January
2,7,July
3,5,May
4,7,July


In [12]:
day_map = {
    1: "Monday",
    2: "Tuesday",
    3: "Wednesday",
    4: "Thursday",
    5: "Friday",
    6: "Saturday",
    7: "Sunday"
}

cleaned_df["DAY_NAME"] = cleaned_df["DAY_OF_WEEK"].map(day_map)

cleaned_df[["DAY_OF_WEEK", "DAY_NAME"]].head()

,DAY_OF_WEEK,DAY_NAME
0,2,Tuesday
1,6,Saturday
2,3,Wednesday
3,2,Tuesday
4,1,Monday


In [13]:
cleaned_df["SCHEDULED_DEPARTURE"] = (
    cleaned_df["SCHEDULED_DEPARTURE"]
    .astype(str)
    .str.zfill(4)
)
cleaned_df["SCHEDULED_DEPARTURE"].head()

0    1340
1    1910
2    0630
3    0810
4    1000
Name: SCHEDULED_DEPARTURE, dtype: str

In [14]:
cleaned_df["DEPARTURE_HOUR"] = (
    cleaned_df["SCHEDULED_DEPARTURE"]
    .str[:2]
    .astype(int)
)

cleaned_df[["SCHEDULED_DEPARTURE", "DEPARTURE_HOUR"]].head()

,SCHEDULED_DEPARTURE,DEPARTURE_HOUR
0,1340,13
1,1910,19
2,0630,6
3,0810,8
4,1000,10


In [15]:
cleaned_df["ROUTE"] = (
    cleaned_df["ORIGIN_AIRPORT"]
    + "-"
    + cleaned_df["DESTINATION_AIRPORT"]
)

cleaned_df[["ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "ROUTE"]].head()

,ORIGIN_AIRPORT,DESTINATION_AIRPORT,ROUTE
0,FWA,DTW,FWA-DTW
1,LAS,SEA,LAS-SEA
2,OAK,SEA,OAK-SEA
3,STL,DAL,STL-DAL
4,IAH,HNL,IAH-HNL


In [17]:
cleaned_df["FLIGHT_DATE"] = pd.to_datetime(
    cleaned_df[["YEAR", "MONTH", "DAY"]]
)

cleaned_df[["YEAR", "MONTH", "DAY", "FLIGHT_DATE"]].head()

,YEAR,MONTH,DAY,FLIGHT_DATE
0,2015,4,7,2015-04-07
1,2015,1,24,2015-01-24
2,2015,7,8,2015-07-08
3,2015,5,26,2015-05-26
4,2015,7,6,2015-07-06


In [18]:
cleaned_df["FLIGHT_DATE"].dtype

dtype('<M8[us]')

In [19]:
cleaned_df["DATE"] = cleaned_df["FLIGHT_DATE"].dt.isocalendar().week
cleaned_df["IS_WEEKEND"] = cleaned_df["DAY_NAME"].isin(["Saturday", "Sunday"])
cleaned_df[["DATE","DAY_NAME","IS_WEEKEND"]].head()

,DATE,DAY_NAME,IS_WEEKEND
0,15,Tuesday,False
1,4,Saturday,True
2,28,Wednesday,False
3,22,Tuesday,False
4,28,Monday,False


In [36]:

cleaned_df.columns

Index(['YEAR', 'MONTH', 'DAY', 'DAY_OF_WEEK', 'AIRLINE', 'FLIGHT_NUMBER',
       'TAIL_NUMBER', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT',
       'SCHEDULED_DEPARTURE', 'DEPARTURE_TIME', 'DEPARTURE_DELAY', 'TAXI_OUT',
       'WHEELS_OFF', 'SCHEDULED_TIME', 'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE',
       'WHEELS_ON', 'TAXI_IN', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME',
       'ARRIVAL_DELAY', 'DIVERTED', 'CANCELLED', 'CANCELLATION_REASON',
       'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY',
       'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY', 'MONTH_NAME', 'DAY_NAME',
       'DEPARTURE_HOUR', 'ROUTE', 'FLIGHT_DATE', 'WEEK', 'IS_WEEKEND'],
      dtype='str')

In [37]:
cleaned_df.to_csv("data/flights_cleaned.csv", index=False)

print("Updated cleaned dataset saved successfully!")

Updated cleaned dataset saved successfully!


In [38]:
test_df = pd.read_csv("data/flights_cleaned.csv")
test_df.head()

C:\Users\Asus\AppData\Local\Temp\ipykernel_19692\479681572.py:1: DtypeWarning: Columns (0: CANCELLATION_REASON) have mixed types. Specify dtype option on import or set low_memory=False.
  test_df = pd.read_csv("data/flights_cleaned.csv")


,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY,MONTH_NAME,DAY_NAME,DEPARTURE_HOUR,ROUTE,FLIGHT_DATE,WEEK,IS_WEEKEND
0,2015,4,7,2,EV,4900,N759EV,FWA,DTW,1340,...,0.0,0.0,0.0,April,Tuesday,13,FWA-DTW,2015-04-07,15,False
1,2015,1,24,6,AS,611,N413AS,LAS,SEA,1910,...,0.0,0.0,0.0,January,Saturday,19,LAS-SEA,2015-01-24,4,True
2,2015,7,8,3,WN,1483,N463WN,OAK,SEA,630,...,0.0,0.0,0.0,July,Wednesday,6,OAK-SEA,2015-07-08,28,False
3,2015,5,26,2,WN,193,N7745A,STL,DAL,810,...,0.0,152.0,0.0,May,Tuesday,8,STL-DAL,2015-05-26,22,False
4,2015,7,6,1,UA,253,N213UA,IAH,HNL,1000,...,0.0,0.0,0.0,July,Monday,10,IAH-HNL,2015-07-06,28,False
